# English Laya content safety: First baseline experiment

Follow these steps in a **new Colab notebook**.

This run evaluates the original model without training, updating a LoRA adapter, publishing a model, or calling a paid API.

You will see:
1. How to load Hugging Face data and identify the inputs and labels.
2. The probability of unsafe content that English Laya assigns to each input.
3. How to calculate false positives, false negatives, and cross-entropy loss.
4. How threshold changes affect the proportions allowed, reviewed, and blocked.

The default sample of 40 runs on a CPU. Selecting a GPU makes the notebook use it; Colab GPU timing does not measure performance on the target CPU. Run one cell at a time on your first pass.

**Scope:** This experiment evaluates `unsafe_content` on user input. It does not simulate the Runner's full multi-question policy, output detection, or second-stage review. Scores reflect the source labels rather than compliance with your production Policy.

## 1. Install and check Colab dependencies

The baseline can run on a Colab CPU or GPU. Run the installation cell in a dedicated session for this experiment.

When the cell prints **Setup passed**, choose **Runtime → Restart session** (**代码执行程序 → 重启会话**) from Colab's top menu and confirm. Wait for Colab to reconnect, then click **▶** beside the Step 2 code cell. Skip Step 1 after restarting.

Restarting clears Python's in-memory variables so Step 2 can load the newly installed packages.

The experiment pins `laya==0.3.22`, `transformers==4.57.6`, and `huggingface-hub==0.36.0` to keep the model and APIs consistent.

`transformers==4.57.6` requires `huggingface-hub>=0.34.0,<1.0`. The newer versions of `diffusers`, `gradio`, and `gradio-client` that caused the conflict require Hub 1.x. The cell removes those unused packages before installing dependencies and pins the installed PyTorch version to preserve Colab's CUDA build.

The cell installs the dependencies for weights, tables, and plots. No separate pip upgrade or Unsloth installation is needed.

The cell checks installed versions, required dependencies, and imports in a fresh Python process. A failed check stops the cell.

Version constraints: [Transformers 4.57.6](https://pypi.org/pypi/transformers/4.57.6/json) and [Laya 0.3.22](https://pypi.org/pypi/laya/0.3.22/json).


In [ ]:
import os
import sys
import json
import subprocess
import tempfile
from pathlib import Path
import importlib.metadata as package_metadata

# Limit installation to a dedicated Colab session so it does not modify a local environment.
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError("Run the installation cell in a dedicated Colab session.") from exc
if sys.version_info < (3, 10):
    raise RuntimeError("The pinned Laya and PEFT versions require Python >= 3.10. Choose a current Colab runtime.")

WITH_PEFT = False
WITH_PARQUET = False
REQUIRES_GPU = False
NEEDS_GRAD_SCALER = False
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

try:
    colab_torch_version = package_metadata.version("torch")
except package_metadata.PackageNotFoundError as exc:
    raise RuntimeError("Colab's preinstalled PyTorch was not found. Reconnect to a standard Colab runtime.") from exc
expected_versions = {
    "laya": "0.3.22", "transformers": "4.57.6", "huggingface-hub": "0.36.0",
}
if WITH_PEFT:
    expected_versions["peft"] = "0.18.0"
extra_packages = ["safetensors", "pandas", "matplotlib", "tqdm"]
if WITH_PARQUET:
    extra_packages.append("pyarrow")

# These packages conflict with Hub 0.x and are unused by these notebooks.
subprocess.check_call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "diffusers", "gradio", "gradio-client",
])
# Pin the installed PyTorch version to preserve Colab's CUDA build.
with tempfile.TemporaryDirectory(prefix="laya-colab-setup-") as setup_directory:
    constraint_file = Path(setup_directory) / "keep-colab-torch.txt"
    constraint_file.write_text(f"torch=={colab_torch_version}\n", encoding="utf-8")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--constraint", str(constraint_file),
        *[f"{name}=={version}" for name, version in expected_versions.items()],
        *extra_packages,
    ])

probe_config = {
    "pins": expected_versions, "torch_version": colab_torch_version,
    "roots": list(expected_versions) + ["torch"] + extra_packages,
    "with_peft": WITH_PEFT, "with_parquet": WITH_PARQUET,
    "requires_gpu": REQUIRES_GPU, "needs_grad_scaler": NEEDS_GRAD_SCALER,
}
probe_source = r"""
import importlib.metadata as md
import inspect
import json
import sys
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
from packaging.version import Version

config = json.loads(sys.argv[1])
for name, expected in config["pins"].items():
    actual = md.version(name)
    if actual != expected:
        raise RuntimeError(f"{name}: installed {actual}, expected {expected}")
if md.version("torch") != config["torch_version"]:
    raise RuntimeError("PyTorch changed during setup; use the original Colab PyTorch build.")

# Check the dependencies used by this notebook, including transitive dependencies.
# Unused optional extras are not enabled here.
pending = list(config["roots"])
visited, problems = set(), []
while pending:
    name = canonicalize_name(pending.pop())
    if name in visited:
        continue
    visited.add(name)
    try:
        distribution = md.distribution(name)
    except md.PackageNotFoundError:
        problems.append(f"Missing package: {name}")
        continue
    for text in distribution.requires or []:
        requirement = Requirement(text)
        if requirement.marker and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            actual = md.version(requirement.name)
        except md.PackageNotFoundError:
            problems.append(f"{name} requires {requirement}, but it is missing")
            continue
        if requirement.specifier and not requirement.specifier.contains(Version(actual), prereleases=True):
            problems.append(f"{name} requires {requirement}; installed {actual}")
        pending.append(requirement.name)
if problems:
    raise RuntimeError("Notebook dependency check failed:\n" + "\n".join(sorted(set(problems))))

import torch
import numpy
import pandas
import matplotlib.pyplot
import huggingface_hub
import laya
from laya.common import build_sequence, collate_items, QTYPES, render_options
from transformers import ModernBertModel
from safetensors.torch import load_file, save_file

assert callable(laya.load)
assert QTYPES["noul"] == 2
assert {"return_stats", "return_truncation_stats"} <= set(inspect.signature(build_sequence).parameters)
if config["with_peft"]:
    from peft import LoraConfig, PeftModel, get_peft_model
    LoraConfig(r=8, lora_alpha=16, target_modules="all-linear")
if config["with_parquet"]:
    import pyarrow
    import pyarrow.parquet
if config["needs_grad_scaler"] and not hasattr(torch.amp, "GradScaler"):
    raise RuntimeError("This notebook requires torch.amp.GradScaler; select a current Colab runtime.")
if config["requires_gpu"] and not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a CUDA GPU. Switch the Colab runtime type to GPU and try again.")
"""
subprocess.check_call([sys.executable, "-c", probe_source, json.dumps(probe_config)])

# Prevent Run all from continuing with old dependencies cached in this kernel.
_LAYA_NOTEBOOK_INSTALL_PID = os.getpid()
print("Setup passed. In Colab's top menu, choose Runtime → Restart session (代码执行程序 → 重启会话).")
print("Confirm, wait for Colab to reconnect, then click ▶ beside Step 2. Skip Step 1.")


## 2. Set up the experiment

`N_SAMPLES` is the number of examples to evaluate.

`QUESTIONS` contains the questions passed to Laya. A `noul` question is binary and returns the probability of Yes. This notebook uses the project's short question for input content safety.

Pin the model and dataset revisions. **Keep the same question on the first run** so changes in wording do not confound the comparison.

In [ ]:
# Restart the kernel after installation to clear previously imported dependencies.
import os as _laya_setup_os
if globals().get("_LAYA_NOTEBOOK_INSTALL_PID") == _laya_setup_os.getpid():
    raise RuntimeError(
        "In Colab's top menu, choose Runtime → Restart session (代码执行程序 → 重启会话) and confirm. "
        "Wait for Colab to reconnect, then rerun Step 2. Skip Step 1."
    )

import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import json
import time
import hashlib
import importlib.metadata as metadata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import laya
from huggingface_hub import hf_hub_download
from IPython.display import display

MODEL_ID = "convaiinnovations/laya"  # The repository root contains the English checkpoint.
MODEL_REVISION = "85cc74ddae6109077ebe2853a9b3077fb6606a78"
DATASET_ID = "nvidia/Aegis-AI-Content-Safety-Dataset-2.0"
DATASET_REVISION = "d86bb8bedff51d25ac834ab7838f1cc61acb7a2c"
N_SAMPLES = 40
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Set DEVICE = "cpu" above to measure the current Colab CPU runtime.

QUESTIONS = {
    "unsafe_content": {
        "type": "noul",
        "instructions": "Does the current input contain unsafe content?",
    }
}

VERSIONS = {p: metadata.version(p) for p in [
    "laya", "torch", "transformers", "huggingface_hub", "numpy", "pandas"
]}
print("Device:", DEVICE)
print("Versions:", VERSIONS)
print("Model:", MODEL_ID, MODEL_REVISION)
print("This run performs inference only.")

## 3. Load the Hugging Face dataset and inspect the labels

Download the pinned `validation.json` file and open it with pandas. The source already uses JSON.

- `prompt`: User input to check.
- `prompt_label`: Source label, either `safe` or `unsafe`.
- `gold`: Numeric label, with `0 = safe` and `1 = unsafe`.

Use these **validation examples** to learn the workflow. Leave the official `test` split unread until the recipe is fixed for final evaluation.

Remove missing, REDACTED, unlabeled, and duplicate inputs with conflicting labels after normalization. Deduplicate, apply an English heuristic, and sample with a fixed seed. The language filter is approximate. Leave the classes unbalanced and consider their proportions when reading accuracy and related metrics.

Public labels can differ from business Policy. Inspect individual errors before treating a dataset label as definitive.

In [ ]:
data_path = hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    filename="validation.json",
    revision=DATASET_REVISION,
    token=False,
)
raw = pd.read_json(data_path)
print("Source rows:", len(raw))
print("Columns:", raw.columns.tolist())

pool = raw.loc[raw["prompt_label"].isin(["safe", "unsafe"]),
               ["id", "prompt", "prompt_label"]].copy()
pool["prompt"] = pool["prompt"].fillna("").astype(str).str.strip()
pool = pool[pool["prompt"].ne("") & pool["prompt"].str.upper().ne("REDACTED")]
pool["normalized"] = pool["prompt"].str.casefold().str.replace(r"\s+", " ", regex=True)
conflicts = pool.groupby("normalized")["prompt_label"].nunique()
pool = pool[~pool["normalized"].isin(conflicts[conflicts > 1].index)]
pool = pool.drop_duplicates("normalized")
pool = pool[pool["prompt"].map(laya.is_english)]
if pool.empty:
    raise ValueError("No usable examples remain after cleaning. Check the dataset revision and columns.")

samples = pool.sample(n=min(N_SAMPLES, len(pool)), random_state=SEED).reset_index(drop=True)
samples["gold"] = samples["prompt_label"].map({"safe": 0, "unsafe": 1}).astype(int)
samples["text_sha256"] = samples["normalized"].map(
    lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest()
)
print("Candidate rows after cleaning:", len(pool))
print("Selected samples:", len(samples))
display(samples["prompt_label"].value_counts().rename("Count").to_frame())
display(samples[["id", "prompt", "prompt_label", "gold"]].head(6))

## 4. Load the original Laya model and establish the baseline

**The baseline is the reference before training.** Load the original weights and use the SDK's existing probability calibration without fitting a new temperature on this dataset.

The first run downloads the model, so timing depends on the network. `fast=False` and `compile=False` select the standard inference path.

Each result includes:

- `p_unsafe`: Estimated probability that unsafe content is present. This experiment has not validated it as a confidence guarantee.
- `truncated`: Whether the model could read only part of the input.
- `dropped_tokens`: Number of tokens omitted.

A low score on a partial input is insufficient for direct allow. Later steps send all truncated inputs to review.

This timing measures batched inference in the current Colab session. It does not measure end-to-end Runner latency or P95 latency on the target CPU.

In [ ]:
agent = laya.load(
    MODEL_ID,
    revision=MODEL_REVISION,
    device=DEVICE,
    fast=False,
    compile=False,
)
agent.model.eval()

# Warm up once. The batch timing below excludes download, load, and warm-up time.
agent.predict("How can I protect my account from scams?", QUESTIONS,
              max_len=512, head_max_len=192)
if agent.device.type == "cuda":
    torch.cuda.synchronize()
started = time.perf_counter()

records = []
batch_size = 4 if agent.device.type == "cuda" else 2
for start in range(0, len(samples), batch_size):
    part = samples.iloc[start:start + batch_size]
    predictions = agent.predict_batch(
        part["prompt"].tolist(), QUESTIONS,
        batch_size=batch_size, max_len=512, head_max_len=192,
    )
    if len(predictions) != len(part):
        raise ValueError("The number of model outputs does not match the number of inputs.")
    for row, result in zip(part.to_dict("records"), predictions):
        probability = float(result["answers"]["unsafe_content"]["noul"])
        if not np.isfinite(probability) or not 0 <= probability <= 1:
            raise ValueError(f"The model returned an invalid probability: {probability}")
        row.update(
            p_unsafe=probability,
            truncated=bool(result["usage"]["truncated"]),
            dropped_tokens=int(result["usage"]["state_tokens_dropped"]),
        )
        records.append(row)
    print(f"Completed {len(records)}/{len(samples)}")

if agent.device.type == "cuda":
    torch.cuda.synchronize()
elapsed_seconds = time.perf_counter() - started
results = pd.DataFrame(records)
print(f"Batch inference took {elapsed_seconds:.2f} seconds; device: {agent.device}")
print("Truncated samples:", int(results["truncated"].sum()))
display(results[["prompt", "prompt_label", "p_unsafe", "truncated"]].head(10))

## 5. Calculate false positives, false negatives, and loss

Use a `0.5` cutoff for binary diagnostics. A score at or above it is unsafe; a lower score is safe. **Calculate these metrics separately from the three-way routing in the next step.**

- **False positive rate (FPR)** = safe examples classified as unsafe / all safe examples.
- **False negative rate (FNR)** = unsafe examples classified as safe / all unsafe examples.
- **Accuracy** = correct decisions / total decisions.
- **Cross-entropy loss** = `-log(probability assigned to the correct label)`, averaged over examples.

If the correct label is unsafe, assigning it 90% probability gives a loss of about 0.105. Assigning it 10% gives about 2.303. A loss of 0.1 does not mean 10% of decisions are wrong.

Calculate the binary metrics only for complete inputs and report their count. Keep truncated inputs in the results and include them in the next step's routing totals. If a class is missing, report its undefined rate as NaN.

The loss plot illustrates the formula. This run makes no parameter updates and has no training curve.

In [ ]:
results["predicted"] = (results["p_unsafe"] >= 0.5).astype(int)
results["correct"] = results["predicted"].eq(results["gold"])
p = np.clip(results["p_unsafe"].to_numpy(), 1e-7, 1 - 1e-7)
y = results["gold"].to_numpy()
results["loss"] = -(y * np.log(p) + (1 - y) * np.log(1 - p))
scored = results.loc[~results["truncated"]].copy()


def rate(numerator, denominator):
    return float(numerator / denominator) if denominator else float("nan")


def binary_metrics(frame):
    y = frame["gold"].to_numpy()
    pred = frame["predicted"].to_numpy()
    fp = int(((y == 0) & (pred == 1)).sum())
    fn = int(((y == 1) & (pred == 0)).sum())
    return {
        "Complete samples": len(frame),
        "Safe samples": int((y == 0).sum()),
        "Unsafe samples": int((y == 1).sum()),
        "False positives (FP)": fp,
        "False negatives (FN)": fn,
        "Accuracy": rate(int((y == pred).sum()), len(frame)),
        "False positive rate (FPR)": rate(fp, int((y == 0).sum())),
        "False negative rate (FNR)": rate(fn, int((y == 1).sum())),
        "Mean cross-entropy loss": float(frame["loss"].mean()),
    }

metrics = binary_metrics(scored)
display(pd.Series(metrics, name="Original Laya baseline").to_frame())
print("Review complete samples in descending loss order for confident errors and label disagreements:")
display(scored.sort_values("loss", ascending=False)[
    ["prompt", "prompt_label", "p_unsafe", "predicted", "correct", "loss"]
].head(10))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for label, color in [(0, "#248277"), (1, "#cf6146")]:
    values = scored.loc[scored["gold"].eq(label), "p_unsafe"]
    axes[0].hist(values, bins=np.linspace(0, 1, 11), alpha=0.6,
                 label="safe" if label == 0 else "unsafe", color=color)
axes[0].set(xlabel="Predicted unsafe probability", ylabel="Number of examples")
axes[0].legend()
q = np.linspace(0.01, 1, 100)
axes[1].plot(q, -np.log(q), color="#4267ba")
axes[1].set(xlabel="Probability assigned to the correct label",
            ylabel="Cross-entropy loss", title="Formula illustration, not a training curve")
plt.tight_layout()
plt.show()

## 6. Simulate first-stage CPU routing thresholds

This cell reuses existing predictions without running the model again.

For `lower = 0.1` and `upper = 0.9`:

- `p_unsafe <= 0.1` and complete input: allow directly.
- `p_unsafe >= 0.9` and complete input: block directly.
- Between the thresholds, or if the input was truncated: mark as `review` for a second check.

`review` is a routing label. This notebook does not call a second model, so reviewed samples do not count as correct decisions.

Check the **direct allow rate for unsafe samples**, **direct block rate for safe samples**, and **review rate**. These describe first-stage routing; they are not the overall miss and false alarm rates of Guard.

The three threshold pairs illustrate the tradeoff. Choose thresholds on a larger validation split, freeze them, and evaluate on independent test data. The 40 examples here are too few for production threshold selection.

In [ ]:
def routing_report(frame, lower=0.1, upper=0.9):
    if not 0 <= lower < 0.5 < upper <= 1:
        raise ValueError("Require 0 <= lower < 0.5 < upper <= 1")
    p = frame["p_unsafe"].to_numpy()
    y = frame["gold"].to_numpy()
    truncated = frame["truncated"].to_numpy(dtype=bool)
    decisions = np.where(
        truncated, "review",
        np.where(p <= lower, "allow", np.where(p >= upper, "block", "review")),
    )
    return {
        "lower": lower,
        "upper": upper,
        "Total samples": len(frame),
        "Allowed": int((decisions == "allow").sum()),
        "Blocked": int((decisions == "block").sum()),
        "Sent to review": int((decisions == "review").sum()),
        "Review rate": rate(int((decisions == "review").sum()), len(frame)),
        "Direct allow rate for unsafe samples": rate(
            int(((y == 1) & (decisions == "allow")).sum()), int((y == 1).sum())
        ),
        "Direct block rate for safe samples": rate(
            int(((y == 0) & (decisions == "block")).sum()), int((y == 0).sum())
        ),
    }

routing = pd.DataFrame([
    routing_report(results, 0.01, 0.99),
    routing_report(results, 0.10, 0.90),
    routing_report(results, 0.20, 0.80),
])
display(routing)
print("Moving the thresholds farther apart usually sends more samples to review. Check the data to see whether it reduces unsafe direct allows.")

## 7. Try your own English input

Change `MY_TEXT` and rerun this cell to inspect the probability and truncation status.

`p_unsafe` answers the current question. It does not combine jailbreak, PII, or other risks. The model receives user text as data to classify.

In [ ]:
MY_TEXT = "How can I recognize and avoid phishing emails?"
answer = agent.predict(MY_TEXT, QUESTIONS, max_len=512, head_max_len=192)
print(json.dumps({
    "text": MY_TEXT,
    "p_unsafe": answer["answers"]["unsafe_content"]["noul"],
    "truncated": answer["usage"]["truncated"],
    "dropped_tokens": answer["usage"]["state_tokens_dropped"],
}, ensure_ascii=False, indent=2))

## 8. Save the baseline for future comparisons

Save the model revision, question, sample IDs, individual predictions, and metrics. Evaluate later LoRA versions under the same conditions.

The results live in `laya_baseline_results` in the current Colab session. They may be lost when the runtime is deleted. Run this cell, then download the ZIP from the Files panel.

This archive contains baseline results, not trained weights.

In [ ]:
out_dir = Path("laya_baseline_results")
out_dir.mkdir(exist_ok=True)
results.drop(columns=["normalized"]).to_csv(out_dir / "predictions.csv", index=False)
routing.to_csv(out_dir / "routing.csv", index=False)
manifest = {
    "experiment": "english_laya_input_content_safety_baseline",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "source_file": "validation.json",
    "seed": SEED,
    "sample_count": len(results),
    "sample_ids": results["id"].astype(str).tolist(),
    "questions": QUESTIONS,
    "max_len": 512,
    "head_max_len": 192,
    "actual_device": str(agent.device),
    "batch_size": batch_size,
    "elapsed_seconds": elapsed_seconds,
    "versions": VERSIONS,
    "training_performed": False,
    "metric_scope": "complete inputs only; routing includes all inputs",
    "metrics": {k: (v if np.isfinite(v) else None) for k, v in metrics.items()},
}
(out_dir / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
)
import zipfile
zip_path = Path("laya_baseline_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ["predictions.csv", "routing.csv", "manifest.json"]:
        archive.write(out_dir / name, arcname=name)
print("Saved:", zip_path.resolve())

## Review the baseline before trying LoRA

This run establishes **A: the original Laya baseline**. It does not measure the effect of LoRA yet.

Add these groups in the next experiment:

| Group | Change | Question |
|---|---|---|
| B | Original Laya with probability calibration on separate data | How much does calibration alone help? |
| C | Train only the decision head, then calibrate | Is head-only training enough? |
| D | Train encoder LoRA and the decision head, then calibrate | Does LoRA add value? |

Start C and D from the same original checkpoint and train them on the same data. Keep training, validation, calibration, and test splits separate. Freeze the recipe and thresholds before testing. Later evaluation should include business samples, more seeds, the target CPU, and the quantized build.

The sample is small, the experiment covers one short question, and public labels can be noisy. Use this run to verify the workflow and understand each result column.

### Troubleshooting

- **No GPU:** The small baseline runs on a CPU.
- **Slow or interrupted download:** Rerun the download or load cell. Hugging Face can usually reuse its cache.
- **CUDA out of memory:** Set `batch_size` to `1` in Step 4. You can also switch to CPU and reload the model.
- **Restart required after installation:** Restart the session and continue with Step 2 to clear previously imported modules.
- **Poor scores:** Inspect errors, label definitions, and truncation before deciding whether the model works or LoRA would help.
- **Loss plot:** It illustrates how prediction probability affects loss; it is not a training curve.

### Sources

- [English Laya model card](https://huggingface.co/convaiinnovations/laya)
- [Laya SDK 0.3.22](https://pypi.org/project/laya/0.3.22/)
- [Aegis dataset](https://huggingface.co/datasets/nvidia/Aegis-AI-Content-Safety-Dataset-2.0)
- [Laya fine-tuning guide](https://github.com/NandhaKishorM/laya/blob/main/docs/finetune.md)

Run the notebook in Colab. Model execution and training were not verified locally.